# ViT 实现


![vit](figures/vit.png)

## 约定：
```py
# 输入图像：x ∈ R^{B×3×H×W}，分类常用 H=W=224
# patch_size = P（P=16）
# patch 数：N = (H/P) * (W/P)（224、P=16 时 N=14*14=196）
# hidden dim：D=768（ViT-Base）
# heads：h=12，每头维度 d = D/h = 64
# depth（encoder blocks）：L=12
# MLP 隐藏维度：D_mlp = 4D = 3072
# tokens 序列长度：T = N + 1（+1 是 CLS）
```


### 0 输入 和 配置

- 输入：`x: [B, 3, 224, 224]`

In [ ]:
# 配置
from typing import Callable, NamedTuple
import torch
import torch.nn as nn

class StemConfig(NamedTuple):
    """Configuration for ViT stem layer.

    Attributes:
        out_channels: Output channel count for this stem layer.
        kernel_size: Kernel size for the convolution.
        stride: Stride for the convolution.
        norm_layer: Normalization layer constructor.
        activation_layer: Activation layer constructor.
    """

    image_size: int     # assumed square 224.
    patch_size: int     # assumed square 16.
    in_channels: int    # usually 3 for RGB images.

    embed_dim: int   # usually 768 for ViT-Base.
    kernel_size: int    # usually equal to patch_size.
    stride: int         # usually equal to patch_size.
    norm: Callable[..., nn.Module] = nn.Identity # optional BatchNorm2d
    activation_layer: Callable[..., nn.Module] = nn.ReLU  # optional

### Layer1 Process Input

#### Layer1.1 Patch Embedding（把图切块变 token）
Patchify + Linear Projection（通常用 Conv2d 实现）

- 操作：把图像按 P×P 切成 patch，并把每个 patch 映射到 D 维

- 等价两种写法：

    1. unfold 后对每个 patch 做 Linear(P*P*3 → D)
    2. 直接 Conv2d(in=3, out=D, kernel=P, stride=P) -> 用的这个

- 输出：
    - patch grid：[B, D, H/P, W/P] = [B, 768, 14, 14]
    - 展平为序列：[B, N, D] = [B, 196, 768]

> 记笔记：patch embedding 的参数量（Conv2d）是 D * 3 * P * P + D(bias)。

In [ ]:
# Layer 1.1: Patch Embedding
class PatchEmbedding(nn.Module):
    def __init__(self, cfg: StemConfig):
        super().__init__()
        self.img_size = cfg.image_size
        self.patch_size = cfg.patch_size
        self.num_patches = (cfg.image_size // cfg.patch_size) ** 2 # 196

        self.proj = nn.Conv2d(
            cfg.in_channels, cfg.embed_dim, kernel_size=cfg.kernel_size, stride=cfg.stride
        ) # B, 3, 224, 224 -> B, 768, 14, 14 
        self.norm = cfg.norm(cfg.embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        torch._assert(H == self.img_size and W == self.img_size, 
                      "Input image size doesn't match model.")
        x = self.proj(x)  # B, 3, 224, 224 -> B, 768, 14, 14
        x = self.norm(x)
        x = x.flatten(2)  # B, 768, 14, 14 -> B, 768, 196
        x = x.transpose(1, 2)  # B, 768, 196 -> B, 196, 768
        
        return x

### Layer1.2 + CLS token

### Layer 1.3 — Add Position Embedding

### Layer2. Transformer Encoder Blocks × L
输入: `[B, T, D]`

#### Layer 2.1 — LN（Pre-LN）

#### Layer 2.2 — MHSA（自注意力）

#### Layer 2.3 — LN（Pre-LN）

#### Layer 2.4 — MLP（FFN） （两层前馈）

### Layer 5 — Final LN 最后 LayerNorm

### Layer 6 — CLS Pooling 取 CLS 做分类

### 7. Layer 7 — Classification Head

In [20]:
# Attention Mask 示例
# 场景：seq_len=5 的序列，需要掩蔽最后一个 token

seq_len = 5
# 创建 attention mask: False = 可以注意, True = 被掩蔽
mask = torch.zeros(seq_len, seq_len, dtype=torch.bool)
mask[-1, :] = True  # 掩蔽最后一行（最后一个 token 无法注意任何位置）

# 或者：因果 mask（causal mask，未来的 token 无法被看到）
causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()

# 应用到 attention scores（通常在 softmax 前）
attention_scores = torch.randn(2, 12, seq_len, seq_len)  # [B, heads, seq_len, seq_len]
attention_scores = attention_scores.masked_fill(causal_mask, float('-inf'))
print("Masked attention scores:\n", attention_scores[0, 0])



Masked attention scores:
 tensor([[-0.2188,    -inf,    -inf,    -inf,    -inf],
        [-1.2903,  0.7951,    -inf,    -inf,    -inf],
        [-0.6968,  0.3610, -1.2876,    -inf,    -inf],
        [-1.6237, -0.5461, -0.0563, -0.5749,    -inf],
        [ 0.3184,  0.4385,  0.9308,  1.5016,  2.1755]])
